In [ ]:
# =========================================================================
# TITLE: REPRODUCIBLE MULTI-SENSOR BIOMASS ESTIMATION ENGINE (PLSR-VIP-RFE)
# SENSORS: Sentinel-1 (SAR C-Band), ALOS-2 (SAR L-Band), Sentinel-2 (MSI Opt.)
# PLATFORM: Google Earth Engine (GEE) + Scikit-Learn (1-SE Rule Optimization)
# =========================================================================

# =========================================================================
# 1. ENVIRONMENT INITIALIZATION & DEPENDENCIES
# =========================================================================
# 1.1 Un-comment if executing in a clean Google Colab runtime:
# !pip install earthengine-api pandas numpy scikit-learn openpyxl scipy

import ee
import math
import numpy as np
import pandas as pd
from scipy.stats import f
from sklearn.preprocessing import StandardScaler
from sklearn.cross_decomposition import PLSRegression
from sklearn.model_selection import RepeatedKFold, KFold
from google.colab import files

# 1.2 Authentication
ee.Authenticate()

# =========================================================================
# 2. USER-DEFINED STUDY CONFIGURATION
# =========================================================================

# -------------------------------------------------------------------------
# 2.1 GEE Infrastructure Settings
# -------------------------------------------------------------------------
# Set your Google Cloud / Google Earth Engine Project ID
GEE_PROJECT_ID = 'your-project-id-here'

# Initialize Earth Engine directly with the project ID
try:
    ee.Initialize(project=GEE_PROJECT_ID)
    print(f" Successfully initialized Google Earth Engine with project: {GEE_PROJECT_ID}")
except Exception as err:
    raise RuntimeError(
        f"\n❌ CRITICAL AUTHENTICATION FAILURE for project '{GEE_PROJECT_ID}':\n"
        f"Details: {err}\n\n"
        "Troubleshooting steps:\n"
        "1. Confirm you are signed in with the Google Account that owns this project.\n"
        "2. Ensure the Earth Engine API is enabled on your Cloud Console:\n"
        f"   https://console.cloud.google.com/apis/library/earthengine.googleapis.com?project={GEE_PROJECT_ID}\n"
        "3. Ensure your account has the 'Earth Engine Resource Viewer/User' role."
    )

# -------------------------------------------------------------------------
# 2.2 Vector Assets Configuration in GEE
# -------------------------------------------------------------------------
# Study area geometry boundary (FeatureCollection from Shapefile/GeoJSON/KML)
# NOTE: Must be a Vector Table Asset, NOT a Raster GeoTIFF.
ASSET_ROI = "projects/your-project-id-here/assets/your_study_area_boundary"

# Ground-truth sample collection (Table Asset uploaded to GEE)
ASSET_POINTS = "projects/your-project-id-here/assets/your_field_sample_points"

# -------------------------------------------------------------------------
# 2.3 Temporal Window Configuration
# -------------------------------------------------------------------------
# Central target date for multi-sensor temporal harmonization (YYYY-MM-DD)
TARGET_DATE_STR = YYYY-MM-DD'
TARGET_DATE = ee.Date(TARGET_DATE_STR)

# Temporal search buffer for Sentinel-1 acquisitions (in days)
S1_WINDOW_DAYS = 6

# Sentinel-2 search window around TARGET_DATE to identify optimal cloud-free scene
S2_DATE_START = 'YYYY-MM-DD'
S2_DATE_END   = 'YYYY-MM-DD'

# -------------------------------------------------------------------------
# 2.4 Ground Data Schema & Ingestion Mapping
# -------------------------------------------------------------------------
# Set to True if GEE ingested your CSV as a single semicolon-delimited string
# Set to False if your Asset already contains separate native tabular columns
USE_SEMICOLON_PARSER = True

# CASE A: If USE_SEMICOLON_PARSER is True, configure string property & indices
MERGED_FIELD_NAME = 'ID;x;y;in-out;alt;cover;biomass'
IDX_ID      = 0  # Position of unique identifier
IDX_LON     = 1  # Position of Longitude (X coordinate)
IDX_LAT     = 2  # Position of Latitude (Y coordinate)
IDX_BIOMASS = 6  # Position of in-situ biomass response value

# CASE B: If USE_SEMICOLON_PARSER is False, provide exact native property names
COL_ID      = 'ID'
COL_LON     = 'x'
COL_LAT     = 'y'
COL_BIOMASS = 'biomass'

# -------------------------------------------------------------------------
# 2.5 Advanced Modeling & Cartography Settings
# -------------------------------------------------------------------------
# Physical upper bound for predicted biomass clamping (prevents unbounded extrapolation)
BIOMASS_CLAMP_MAX = 6000.0

# Number of thematic classes for Jenks natural breaks classification
N_CLASSES = 5

# Maximum number of decorrelated features evaluated during RFE 1-SE screening
MAX_PREDICTORS_EVAL = 20

# Master pseudo-random seed for deterministic cross-validation splits
RANDOM_STATE = 42

# =========================================================================
# 3. SPATIAL ASSET INGESTION & DATA CLEANING
# =========================================================================
roi = ee.FeatureCollection(ASSET_ROI)
raw_points = ee.FeatureCollection(ASSET_POINTS)
dem = ee.Image('USGS/SRTMGL1_003').clip(roi)

# Standardized internal field tags
PROP_ID = 'plot_id'
PROP_LON = 'coord_x'
PROP_LAT = 'coord_y'
PROP_TARGET = 'biomass_target'

def parse_point_records(feature):
    """
    Standardizes point geometries and attribute schemas regardless
    of whether inputs are native tables or semicolon-delimited text records.
    """
    if USE_SEMICOLON_PARSER:
        raw_text = ee.String(feature.get(MERGED_FIELD_NAME))
        tokens = raw_text.split(';')

        id_val = tokens.get(IDX_ID)
        lon_val = ee.Number.parse(ee.String(tokens.get(IDX_LON)))
        lat_val = ee.Number.parse(ee.String(tokens.get(IDX_LAT)))
        biomass_val = ee.Number.parse(ee.String(tokens.get(IDX_BIOMASS)))

        geom = ee.Geometry.Point([lon_val, lat_val])
        return ee.Feature(geom, {
            PROP_ID: id_val,
            PROP_LON: lon_val,
            PROP_LAT: lat_val,
            PROP_TARGET: biomass_val
        })
    else:
        geom = feature.geometry()
        return ee.Feature(geom, {
            PROP_ID: feature.get(COL_ID),
            PROP_LON: ee.Number(feature.get(COL_LON)),
            PROP_LAT: ee.Number(feature.get(COL_LAT)),
            PROP_TARGET: ee.Number(feature.get(COL_BIOMASS))
        })

cleaned_points = raw_points.map(parse_point_records)

# =========================================================================
# 4. SENSOR-SPECIFIC PRE-PROCESSING PIPELINES
# =========================================================================
# 4.1 Topographical Attribute Generation (SRTM DEM)
terrain = ee.Terrain.products(dem)
slope_10m = terrain.select('slope').rename('slope').resample('bilinear')
aspect_10m = terrain.select('aspect').rename('aspect').resample('bilinear')
elevation_10m = dem.rename('elevation').resample('bilinear')

# 4.2 Optical Sentinel-2 Processing (Cloud Masking & Topographic C-Correction)
target_year = TARGET_DATE.get('year').getInfo()
s2_coll_id = 'COPERNICUS/S2_HARMONIZED' if target_year < 2018 else 'COPERNICUS/S2_SR_HARMONIZED'

def mask_s2_clouds(image):
    qa = image.select('QA60')
    cloud_bit = 1 << 10
    cirrus_bit = 1 << 11
    mask = qa.bitwiseAnd(cloud_bit).eq(0).And(qa.bitwiseAnd(cirrus_bit).eq(0))
    bands = image.select(['B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B11', 'B12']).multiply(0.0001)
    return image.updateMask(mask).addBands(bands, None, True)

s2_coll = ee.ImageCollection(s2_coll_id) \
    .filterBounds(roi) \
    .filterDate(S2_DATE_START, S2_DATE_END) \
    .map(mask_s2_clouds)

# Fallback in case extreme cloud masking empties the collection
if s2_coll.size().getInfo() == 0:
    s2_coll = ee.ImageCollection(s2_coll_id).filterBounds(roi).filterDate(S2_DATE_START, S2_DATE_END)

s2_scene = s2_coll.map(lambda img: img.set('dist', ee.Number(img.get('system:time_start')).subtract(TARGET_DATE.millis()).abs())) \
    .sort('dist') \
    .first() \
    .clip(roi)

# Native coordinate reference system for raster harmonizations
ref_proj = s2_scene.select('B2').projection()

# Illumination modeling for C-Correction
zenith_rad = ee.Number(s2_scene.get('MEAN_SOLAR_ZENITH_ANGLE')).multiply(math.pi / 180.0)
azimuth_rad = ee.Number(s2_scene.get('MEAN_SOLAR_AZIMUTH_ANGLE')).multiply(math.pi / 180.0)
slope_rad = slope_10m.multiply(math.pi / 180.0)
aspect_rad = aspect_10m.multiply(math.pi / 180.0)

cos_i = slope_rad.cos().multiply(zenith_rad.cos()) \
    .add(slope_rad.sin().multiply(zenith_rad.sin()).multiply(aspect_rad.subtract(azimuth_rad).cos())) \
    .rename('cosi')
cos_theta = zenith_rad.cos()

optical_bands_corrected = []
for b_name in ['B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B11', 'B12']:
    raw_b = s2_scene.select(b_name)
    fit = cos_i.addBands(raw_b).reduceRegion(
        reducer=ee.Reducer.linearFit(),
        geometry=roi.geometry().bounds(),
        scale=60,
        maxPixels=1e9
    )
    b_val = ee.Number(ee.List([fit.get('scale'), 0]).reduce(ee.Reducer.firstNonNull()))
    a_val = ee.Number(ee.List([fit.get('offset'), 0]).reduce(ee.Reducer.firstNonNull()))
    is_valid = b_val.gt(0.00001)
    c_factor = ee.Number(ee.Algorithms.If(is_valid, a_val.divide(ee.Algorithms.If(is_valid, b_val, 1.0)), 0))
    corr_band = raw_b.multiply(cos_theta.add(c_factor)).divide(cos_i.add(c_factor)).rename(b_name)
    optical_bands_corrected.append(corr_band)

s2_corrected = ee.Image.cat(optical_bands_corrected)

# 4.3 Radar Sentinel-1 SAR (Terrain Normalization & Refined Lee Filter)
def process_s1_frame(img):
    vv_lin = ee.Image(10).pow(img.select('VV').divide(10)).rename('VV_lin')
    vh_lin = ee.Image(10).pow(img.select('VH').divide(10)).rename('VH_lin')

    t_sar = ee.Algorithms.Terrain(dem)
    slp = t_sar.select('slope').multiply(math.pi / 180.0)
    asp = t_sar.select('aspect').multiply(math.pi / 180.0)
    theta_inc = img.select('angle').multiply(math.pi / 180.0)
    azimuth_sar = ee.Number(258).multiply(math.pi / 180.0)

    cos_theta_loc = slp.cos().multiply(theta_inc.cos()) \
        .add(slp.sin().multiply(theta_inc.sin()).multiply(asp.subtract(azimuth_sar).cos()))

    # Division lower-bound clamp prevents NaN generation in complex terrain
    g0_vv = vv_lin.multiply(theta_inc.cos().divide(cos_theta_loc.abs().max(0.1))).rename('VV_g0')
    g0_vh = vh_lin.multiply(theta_inc.cos().divide(cos_theta_loc.abs().max(0.1))).rename('VH_g0')

    # Non-destructive angle filtering
    valid_mask = img.select('angle').gt(25).And(img.select('angle').lt(50))
    return img.addBands([g0_vv, g0_vh]).select(['VV_g0', 'VH_g0']).updateMask(valid_mask)

def apply_refined_lee(image, bands, suffix):
    stats_mean = image.select(bands).reduceNeighborhood(ee.Reducer.mean(), ee.Kernel.square(5, 'pixels'))
    stats_var = image.select(bands).reduceNeighborhood(ee.Reducer.variance(), ee.Kernel.square(5, 'pixels'))
    noise_var = 1.0 / 4.9

    filtered_bands = []
    for b in bands:
        b_var = stats_var.select(f"{b}_variance")
        b_mean = stats_mean.select(f"{b}_mean")
        k_factor = b_var.divide(b_var.add(b_mean.pow(2).multiply(noise_var)).max(0.00001))
        filt = b_mean.add(k_factor.multiply(image.select(b).subtract(b_mean))).rename(f"{b}_{suffix}")
        filtered_bands.append(filt)
    return image.addBands(filtered_bands)

# Multi-orbit collection ingestion (supports ascending and descending passes)
s1_coll = ee.ImageCollection('COPERNICUS/S1_GRD') \
    .filterBounds(roi) \
    .filter(ee.Filter.eq('instrumentMode', 'IW')) \
    .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV')) \
    .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH')) \
    .filterDate(TARGET_DATE.advance(-S1_WINDOW_DAYS, 'day'), TARGET_DATE.advance(S1_WINDOW_DAYS, 'day'))

# Automatic window expansion if satellite revisit is low
if s1_coll.size().getInfo() == 0:
    print("⚠️ Revisit coverage alert: Expanding S1 temporal search window to 30 days...")
    s1_coll = ee.ImageCollection('COPERNICUS/S1_GRD') \
        .filterBounds(roi) \
        .filter(ee.Filter.eq('instrumentMode', 'IW')) \
        .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV')) \
        .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH')) \
        .filterDate(TARGET_DATE.advance(-15, 'day'), TARGET_DATE.advance(15, 'day'))

s1_linear = s1_coll.map(process_s1_frame).median().clip(roi)
s1_filtered = apply_refined_lee(s1_linear, ['VV_g0', 'VH_g0'], 'lee')

vv_lin_final = s1_filtered.select('VV_g0_lee').rename('VV_lin')
vh_lin_final = s1_filtered.select('VH_g0_lee').rename('VH_lin')
vv_db = vv_lin_final.log10().multiply(10).clamp(-25, -2).rename('VV_dB')
vh_db = vh_lin_final.log10().multiply(10).clamp(-30, -5).rename('VH_dB')

# 4.4 Radar ALOS-2 PALSAR-2 L-Band Calibration & Filtering
alos_coll = ee.ImageCollection('JAXA/ALOS/PALSAR/YEARLY/SAR') \
    .filter(ee.Filter.calendarRange(target_year, target_year, 'year')) \
    .filterBounds(roi)

# Fallback to closest available annual mosaic if current year is not yet published
if alos_coll.size().getInfo() == 0:
    alos_img = ee.Image(ee.ImageCollection('JAXA/ALOS/PALSAR/YEARLY/SAR').filterBounds(roi).sort('system:time_start', False).first())
else:
    alos_img = ee.Image(alos_coll.first())

def calibrate_alos(img):
    scale_factor = ee.Image.constant(10).pow(-8.3)
    hh_lin = img.select('HH').pow(2).multiply(scale_factor).rename('HH_lin')
    hv_lin = img.select('HV').pow(2).multiply(scale_factor).rename('HV_lin')
    return img.addBands([hh_lin, hv_lin])

alos_linear = calibrate_alos(alos_img).select(['HH_lin', 'HV_lin']).resample('bilinear').clip(roi)
alos_filtered = apply_refined_lee(alos_linear, ['HH_lin', 'HV_lin'], 'lee')

hh_lin_final = alos_filtered.select('HH_lin_lee').rename('HH_lin')
hv_lin_final = alos_filtered.select('HV_lin_lee').rename('HV_lin')
hh_db = hh_lin_final.log10().multiply(10).clamp(-25, -2).rename('HH_dB')
hv_db = hv_lin_final.log10().multiply(10).clamp(-30, -10).rename('HV_dB')

# =========================================================================
# 5. MULTI-SENSOR HYPERCUBE INTEGRATION
# =========================================================================
# 5.1 Formulation of Optical Feature Space
b2, b3, b4, b5, b6, b7, b8, b8a, b11, b12 = [s2_corrected.select(b) for b in ['B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B11', 'B12']]
opt_map = {'B2': b2, 'B3': b3, 'B4': b4, 'B5': b5, 'B6': b6, 'B7': b7, 'B8': b8, 'B8a': b8a, 'B11': b11, 'B12': b12}

def opt_expr(formula, name):
    return s2_corrected.expression(formula, opt_map).rename(name)

optical_index_list = [
    opt_expr('(B8 - B4) / (B8 + B4)', 'NDVI842'),
    opt_expr('(B8a - B4) / (B8a + B4)', 'NDVI865'),
    opt_expr('(B5 - B4) / (B5 + B4)', 'NDVI705'),
    opt_expr('(B6 - B4) / (B6 + B4)', 'NDVI740'),
    opt_expr('(B7 - B4) / (B7 + B4)', 'NDVI783'),
    opt_expr('(B11 - B4) / (B11 + B4)', 'NDVI1610'),
    opt_expr('(B12 - B4) / (B12 + B4)', 'NDVI2190'),
    opt_expr('(B8 - B5) / (B8 + B5)', 'NDRE852_705'),
    opt_expr('(B8 - B6) / (B8 + B6)', 'NDRE852_740'),
    opt_expr('(B8 - B7) / (B8 + B7)', 'NDRE852_783'),
    opt_expr('(B8a - B5) / (B8a + B5)', 'NDRE865'),
    opt_expr('(B8a - B6) / (B8a + B6)', 'NDRE865_740'),
    opt_expr('(B8a - B7) / (B8a + B7)', 'NDRE865_783'),
    b8.subtract(b4).rename('DVI842'),
    b8a.subtract(b4).rename('DVI865'),
    b5.subtract(b4).rename('DVI705'),
    b6.subtract(b4).rename('DVI740'),
    b7.subtract(b4).rename('DVI783'),
    b11.subtract(b4).rename('DVI1610'),
    b12.subtract(b4).rename('DVI2190'),
    opt_expr('2.5 * ((B8 - B4) / (B8 + (B4 * 6) - (B2 * 7.5) + 1))', 'EVI842'),
    opt_expr('2.5 * ((B8a - B4) / (B8a + (B4 * 6) - (B2 * 7.5) + 1))', 'EVI865'),
    opt_expr('2.5 * ((B5 - B4) / (B5 + (B4 * 6) - (B2 * 7.5) + 1))', 'EVI705'),
    opt_expr('2.5 * ((B6 - B4) / (B6 + (B4 * 6) - (B2 * 7.5) + 1))', 'EVI740'),
    opt_expr('2.5 * ((B7 - B4) / (B7 + (B4 * 6) - (B2 * 7.5) + 1))', 'EVI783'),
    opt_expr('2.5 * ((B11 - B4) / (B11 + (B4 * 6) - (B2 * 7.5) + 1))', 'EVI1610'),
    opt_expr('2.5 * ((B12 - B4) / (B12 + (B4 * 6) - (B2 * 7.5) + 1))', 'EVI2190'),
    opt_expr('(B8 - B3) / (B8 + B3)', 'GNDVI842'),
    opt_expr('(B8a - B3) / (B8a + B3)', 'GNDVI865'),
    opt_expr('(B5 - B3) / (B5 + B3)', 'GNDVI705'),
    opt_expr('(B6 - B3) / (B6 + B3)', 'GNDVI740'),
    opt_expr('(B7 - B3) / (B7 + B3)', 'GNDVI783'),
    opt_expr('(B11 - B3) / (B11 + B3)', 'GNDVI1610'),
    opt_expr('(B12 - B3) / (B12 + B3)', 'GNDVI2190'),
    opt_expr('(B8 - B4) / (B8 + B4 + 0.16)', 'OSAVI842'),
    opt_expr('(B8a - B4) / (B8a + B4 + 0.16)', 'OSAVI865'),
    opt_expr('(B5 - B4) / (B5 + B4 + 0.16)', 'OSAVI705'),
    opt_expr('(B6 - B4) / (B6 + B4 + 0.16)', 'OSAVI740'),
    opt_expr('(B7 - B4) / (B7 + B4 + 0.16)', 'OSAVI783'),
    opt_expr('(B11 - B4) / (B11 + B4 + 0.16)', 'OSAVI1610'),
    opt_expr('(B12 - B4) / (B12 + B4 + 0.16)', 'OSAVI2190'),
    opt_expr('((B8 / B4) - 1) / (sqrt(B8 / B4) + 1)', 'MSR842'),
    opt_expr('((B8a / B4) - 1) / (sqrt(B8a / B4) + 1)', 'MSR865'),
    opt_expr('((B5 / B4) - 1) / (sqrt(B5 / B4) + 1)', 'MSR705'),
    opt_expr('((B6 / B4) - 1) / (sqrt(B6 / B4) + 1)', 'MSR740'),
    opt_expr('((B7 / B4) - 1) / (sqrt(B7 / B4) + 1)', 'MSR783'),
    opt_expr('((B11 / B4) - 1) / (sqrt(B11 / B4) + 1)', 'MSR1610'),
    opt_expr('((B12 / B4) - 1) / (sqrt(B12 / B4) + 1)', 'MSR2190'),
    opt_expr('(B8 - B11) / (B8 + B11)', 'NDWI842_1610'),
    opt_expr('(B8a - B11) / (B8a + B11)', 'NDWI865_1610'),
    opt_expr('(B5 - B11) / (B5 + B11)', 'NDWI705_1610'),
    opt_expr('(B6 - B11) / (B6 + B11)', 'NDWI740_1610'),
    opt_expr('(B7 - B11) / (B7 + B11)', 'NDWI783_1610'),
    opt_expr('(B8 - B12) / (B8 + B12)', 'NDWI842_2190'),
    opt_expr('(B8a - B12) / (B8a + B12)', 'NDWI865_2190'),
    opt_expr('(B5 - B12) / (B5 + B12)', 'NDWI705_2190'),
    opt_expr('(B6 - B12) / (B6 + B12)', 'NDWI740_2190'),
    opt_expr('(B7 - B12) / (B7 + B12)', 'NDWI783_2190'),
    opt_expr('(B8 - (0.74 * B4 + 0.26 * B11)) / (B8 + 0.74 * B4 + 0.26 * B11)', 'NDPI842_1610'),
    opt_expr('(B8a - (0.74 * B4 + 0.26 * B11)) / (B8a + 0.74 * B4 + 0.26 * B11)', 'NDPI865_1610'),
    opt_expr('(B5 - (0.74 * B4 + 0.26 * B11)) / (B5 + 0.74 * B4 + 0.26 * B11)', 'NDPI705_1610'),
    opt_expr('(B6 - (0.74 * B4 + 0.26 * B11)) / (B6 + 0.74 * B4 + 0.26 * B11)', 'NDPI740_1610'),
    opt_expr('(B7 - (0.74 * B4 + 0.26 * B11)) / (B7 + 0.74 * B4 + 0.26 * B11)', 'NDPI783_1610'),
    opt_expr('(B8 - (0.74 * B4 + 0.26 * B12)) / (B8 + 0.74 * B4 + 0.26 * B12)', 'NDPI842_2190'),
    opt_expr('(B8a - (0.74 * B4 + 0.26 * B12)) / (B8a + 0.74 * B4 + 0.26 * B12)', 'NDPI865_2190'),
    opt_expr('(B5 - (0.74 * B4 + 0.26 * B12)) / (B5 + 0.74 * B4 + 0.26 * B12)', 'NDPI705_2190'),
    opt_expr('(B6 - (0.74 * B4 + 0.26 * B12)) / (B6 + 0.74 * B4 + 0.26 * B12)', 'NDPI740_2190'),
    opt_expr('(B7 - (0.74 * B4 + 0.26 * B12)) / (B7 + 0.74 * B4 + 0.26 * B12)', 'NDPI783_2190'),
    opt_expr('B8 / (B5 + B11)', 'SLAVI842_1610'),
    opt_expr('B8a / (B5 + B11)', 'SLAVI865_1610'),
    opt_expr('B5 / (B5 + B11)', 'SLAVI705_1610'),
    opt_expr('B6 / (B5 + B11)', 'SLAVI740_1610'),
    opt_expr('B7 / (B5 + B11)', 'SLAVI783_1610'),
    opt_expr('B8 / (B5 + B12)', 'SLAVI842_2190'),
    opt_expr('B8a / (B5 + B12)', 'SLAVI865_2190'),
    opt_expr('B5 / (B5 + B12)', 'SLAVI705_2190'),
    opt_expr('B6 / (B5 + B12)', 'SLAVI740_2190'),
    opt_expr('B7 / (B5 + B12)', 'SLAVI783_2190'),
    b8.divide(b4).rename('SRVI842'),
    b8a.divide(b4).rename('SRVI865'),
    b5.divide(b4).rename('SRVI705'),
    b6.divide(b4).rename('SRVI740'),
    b7.divide(b4).rename('SRVI783'),
    b11.divide(b4).rename('SRVI1610'),
    b12.divide(b4).rename('SRVI2190'),
    opt_expr('((B8 - B4) - 0.2 * (B8 - B3)) * (B8 / B4)', 'MCARI842'),
    opt_expr('((B8a - B4) - 0.2 * (B8a - B3)) * (B8a / B4)', 'MCARI865'),
    opt_expr('((B5 - B4) - 0.2 * (B5 - B3)) * (B5 / B4)', 'MCARI705'),
    opt_expr('((B6 - B4) - 0.2 * (B6 - B3)) * (B6 / B4)', 'MCARI740'),
    opt_expr('((B7 - B4) - 0.2 * (B7 - B3)) * (B7 / B4)', 'MCARI783'),
    opt_expr('((B11 - B4) - 0.2 * (B11 - B3)) * (B11 / B4)', 'MCARI1610'),
    opt_expr('((B12 - B4) - 0.2 * (B12 - B3)) * (B12 / B4)', 'MCARI2190'),
    b8.divide(b3).subtract(1).rename('CGM842'),
    b8a.divide(b3).subtract(1).rename('CGM865'),
    b5.divide(b3).subtract(1).rename('CGM705'),
    b6.divide(b3).subtract(1).rename('CGM740'),
    b7.divide(b3).subtract(1).rename('CGM783'),
    b11.divide(b3).subtract(1).rename('CGM1610'),
    b12.divide(b3).subtract(1).rename('CGM2190'),
    opt_expr('sqrt(0.5 + (B8 - B4) / (B8 + B4))', 'TDVI842'),
    opt_expr('sqrt(0.5 + (B8a - B4) / (B8a + B4))', 'TDVI865'),
    opt_expr('sqrt(0.5 + (B5 - B4) / (B5 + B4))', 'TDVI705'),
    opt_expr('sqrt(0.5 + (B6 - B4) / (B6 + B4))', 'TDVI740'),
    opt_expr('sqrt(0.5 + (B7 - B4) / (B7 + B4))', 'TDVI783'),
    opt_expr('sqrt(0.5 + (B11 - B4) / (B11 + B4))', 'TDVI1610'),
    opt_expr('sqrt(0.5 + (B12 - B4) / (B12 + B4))', 'TDVI2190')
]
image_optical = s2_corrected.addBands(optical_index_list)

# 5.2 Formulation of Radar Feature Space (Dual-Sensor C & L-Band)
rad_map = {'VV': vv_lin_final, 'VH': vh_lin_final, 'HH': hh_lin_final, 'HV': hv_lin_final}

def rad_expr(formula, name):
    return vv_lin_final.expression(formula, rad_map).rename(name)

radar_index_list = [
    rad_expr('(VV - VH) / (VV + VH)', 'RNDI'),
    rad_expr('(HV - VH) / (HV + VH)', 'Cross_NDPI_CL'),
    rad_expr('(VV - HH) / (VV + HH)', 'Co_NDPI_CL'),
    rad_expr('(4 * VH) / (VV + VH)', 'RVIc'),
    rad_expr('(4 * HV) / (HH + HV)', 'RVIl'),
    rad_expr('(4 * VH) / (HH + VH)', 'RVIC_L1'),
    rad_expr('(4 * HV) / (VV + HV)', 'RVIC_L2'),
    rad_expr('(8 * HV) / (HH + VV + (2 * HV))', 'RVIkim'),
    rad_expr('(HH + VV) / 2', 'BMI'),
    rad_expr('VV / VH', 'Ratio_VV_VH'),
    rad_expr('VH / VV', 'Ratio_VH_VV'),
    rad_expr('HH / HV', 'Ratio_HH_HV'),
    rad_expr('HV / HH', 'Ratio_HV_HH'),
    rad_expr('VV / HH', 'Ratio_VV_HH'),
    rad_expr('VH / HV', 'Ratio_VH_HV'),
    rad_expr('VV * VH * HH * HV', 'Mult_All'),
    rad_expr('VV * VH', 'Mult_Co_S1'),
    rad_expr('HH * HV', 'Mult_Co_ALOS'),
    rad_expr('VH * HV', 'Mult_Cross'),
    rad_expr('VV + VH + HH + HV', 'Sum_All'),
    rad_expr('VV + VH', 'Sum_Co_S1'),
    rad_expr('HH + HV', 'Sum_Co_ALOS'),
    rad_expr('VH + HV', 'Sum_Cross'),
    rad_expr('VV - VH', 'Diff_Co_S1'),
    rad_expr('HH - HV', 'Diff_Co_ALOS'),
    rad_expr('VH - HV', 'Diff_Cross')
]
image_radar = ee.Image.cat([vv_db, vh_db, hh_db, hv_db]).addBands(radar_index_list)

# 5.3 Consolidated Master Hypercube
image_topo = ee.Image.cat([elevation_10m, slope_10m, aspect_10m])
master_hypercube = ee.Image.cat([image_optical, image_radar, image_topo]).reproject(crs=ref_proj, scale=10)

optical_features = image_optical.bandNames().getInfo()
radar_features = image_radar.bandNames().getInfo()
fused_features = master_hypercube.bandNames().getInfo()

# =========================================================================
# 6. SPATIAL DATA EXTRACTION TO PYTHON
# =========================================================================
print(f"Total in-situ points registered in collection: {cleaned_points.size().getInfo()}")

# Resilient sampling across the multi-sensor cube
sample_payload = master_hypercube.sampleRegions(
    collection=cleaned_points,
    properties=[PROP_ID, PROP_LON, PROP_LAT, PROP_TARGET],
    scale=10,
    geometries=True,
    tileScale=4
).getInfo()

features_raw = [rec['properties'] for rec in sample_payload['features']]
print(f"Successfully extracted valid intersecting pixel points: {len(features_raw)}")

if len(features_raw) == 0:
    first_pt = cleaned_points.first().geometry().coordinates().getInfo()
    roi_extents = roi.geometry().bounds().getInfo()['coordinates'][0]
    raise ValueError(
        f"❌ SPATIAL INTERSECTION ERROR: 0 sample points intersected the imagery cube!\n"
        f"Sample Point Coordinates: {first_pt}\n"
        f"Study Area ROI Bounds: {roi_extents}\n"
        f"Verify coordinate ordering [Longitude, Latitude] or verify that the points lie within ASSET_ROI."
    )

df_raw = pd.DataFrame(features_raw)
y_all = df_raw[PROP_TARGET].values
x_coords = df_raw[PROP_LON].values
y_coords = df_raw[PROP_LAT].values

# Design matrix of predictors
df_features = df_raw.drop(columns=[PROP_ID, PROP_LON, PROP_LAT, PROP_TARGET], errors='ignore')

# =========================================================================
# 7. MACHINE LEARNING ENGINE (VIP, DECORRELATION, RFE & 1-SE RULE)
# =========================================================================
def calculate_vip(pls_model):
    """Computes Variable Importance in Projection (VIP) metric."""
    t = pls_model.x_scores_
    w = pls_model.x_weights_
    q = pls_model.y_loadings_
    p_vars, a_comps = w.shape
    ssy = np.zeros(a_comps)
    for a in range(a_comps):
        ssy[a] = np.dot(t[:, a], t[:, a]) * (q[0, a] ** 2)
    ssy_total = np.sum(ssy)

    vip_scores = np.zeros(p_vars)
    for j in range(p_vars):
        sum_val = 0.0
        for a in range(a_comps):
            w_norm = w[:, a] / np.linalg.norm(w[:, a])
            sum_val += (w_norm[j] ** 2) * ssy[a]
        vip_scores[j] = np.sqrt(p_vars * sum_val / ssy_total) if ssy_total > 0 else 0.0
    return vip_scores

def select_decorrelated_vip_features(x_scaled, y_train, feat_names, corr_thresh=0.85):
    """Sorts predictors by VIP from a 2-comp PLS pilot and iteratively removes collinears."""
    pls_pilot = PLSRegression(n_components=min(2, x_scaled.shape[1]))
    pls_pilot.fit(x_scaled, y_train)
    vips = calculate_vip(pls_pilot)

    sorted_idx = np.argsort(vips)[::-1]
    ranked_names = [feat_names[i] for i in sorted_idx]

    df_temp = pd.DataFrame(x_scaled, columns=feat_names)
    pruned = []
    for f_name in ranked_names:
        if len(pruned) == 0:
            pruned.append(f_name)
        else:
            corrs = df_temp[pruned].corrwith(df_temp[f_name]).abs()
            if (corrs < corr_thresh).all():
                pruned.append(f_name)
    return pruned, vips

def compute_regression_metrics(y_true, y_pred, k_dim):
    """Computes standard spatial regression statistics."""
    n_samples = len(y_true)
    res = y_true - y_pred
    rmse = np.sqrt(np.mean(res**2))
    mae = np.mean(np.abs(res))
    pbias = 100.0 * np.sum(res) / np.sum(y_true)
    ss_tot = np.sum((y_true - np.mean(y_true))**2)
    r2 = 1.0 - (np.sum(res**2) / ss_tot) if ss_tot > 0 else 0.0

    if r2 > 0 and (n_samples - k_dim - 1) > 0:
        f_stat = (r2 / k_dim) / ((1.0 - r2) / (n_samples - k_dim - 1))
        pval = 1.0 - f.cdf(f_stat, k_dim, n_samples - k_dim - 1)
    else:
        pval = 1.0
    return rmse, mae, pbias, r2, pval

def execute_pipeline(subset_feature_names, model_tag):
    """Executes nested Cross-Validation, RFE-1SE optimization, and Final Model Fit."""
    print(f"\n=======================================================")
    print(f"🚀 Executing Pipeline: {model_tag} ({len(subset_feature_names)} Candidate Features)")
    print(f"=======================================================")
    x_matrix = df_features[subset_feature_names].values

    # Outer Cross Validation (10 Repeats x 5 Folds)
    rkf = RepeatedKFold(n_splits=5, n_repeats=10, random_state=RANDOM_STATE)
    cv_records = []
    fold_idx = 1

    for tr_idx, te_idx in rkf.split(x_matrix, y_all):
        x_tr, x_te = x_matrix[tr_idx], x_matrix[te_idx]
        y_tr, y_te = y_all[tr_idx], y_all[te_idx]

        sc = StandardScaler()
        x_tr_sc = sc.fit_transform(x_tr)
        x_te_sc = sc.transform(x_te)

        # Local non-leaking feature selection
        pruned_fold, _ = select_decorrelated_vip_features(x_tr_sc, y_tr, subset_feature_names)
        max_k_fold = min(MAX_PREDICTORS_EVAL, len(pruned_fold))

        # Internal fold optimization for k
        inner_cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
        inner_r2_means, inner_r2_ses = [], []

        for k in range(1, max_k_fold + 1):
            sub_feats = pruned_fold[:k]
            sub_cols = [subset_feature_names.index(f_name) for f_name in sub_feats]
            x_tr_k = x_tr_sc[:, sub_cols]

            n_comps_inner = 1 if k == 1 else (2 if k <= 3 else 3)
            inner_scores = []
            for in_tr, in_val in inner_cv.split(x_tr_k):
                pls_in = PLSRegression(n_components=n_comps_inner)
                pls_in.fit(x_tr_k[in_tr], y_tr[in_tr])
                inner_scores.append(pls_in.score(x_tr_k[in_val], y_tr[in_val]))
            inner_r2_means.append(np.mean(inner_scores))
            inner_r2_ses.append(np.std(inner_scores) / np.sqrt(len(inner_scores)))

        idx_opt = np.argmax(inner_r2_means)
        cutoff = inner_r2_means[idx_opt] - inner_r2_ses[idx_opt]
        k_opt_fold = [idx + 1 for idx, score in enumerate(inner_r2_means) if score >= cutoff][0]

        # Internal component optimization
        sub_feats_final = pruned_fold[:k_opt_fold]
        sub_cols_final = [subset_feature_names.index(f_name) for f_name in sub_feats_final]
        x_tr_opt = x_tr_sc[:, sub_cols_final]
        x_te_opt = x_te_sc[:, sub_cols_final]

        best_comp, best_score = 1, -np.inf
        for c in range(1, min(k_opt_fold, 5) + 1):
            c_scores = [PLSRegression(n_components=c).fit(x_tr_opt[in_tr], y_tr[in_tr]).score(x_tr_opt[in_val], y_tr[in_val])
                        for in_tr, in_val in inner_cv.split(x_tr_opt)]
            if np.mean(c_scores) > best_score:
                best_score = np.mean(c_scores)
                best_comp = c

        pls_outer = PLSRegression(n_components=best_comp)
        pls_outer.fit(x_tr_opt, y_tr)

        m_tr = compute_regression_metrics(y_tr, pls_outer.predict(x_tr_opt).flatten(), k_opt_fold)
        m_te = compute_regression_metrics(y_te, pls_outer.predict(x_te_opt).flatten(), 1)

        cv_records.append({
            'Fold': fold_idx, 'k_opt': k_opt_fold, 'Components': best_comp,
            'Train_RMSE': m_tr[0], 'Train_MAE': m_tr[1], 'Train_PBIAS': m_tr[2], 'Train_R2': m_tr[3], 'Train_pval': m_tr[4],
            'Test_RMSE': m_te[0], 'Test_MAE': m_te[1], 'Test_PBIAS': m_te[2], 'Test_R2': m_te[3], 'Test_pval': m_te[4]
        })
        fold_idx += 1

    df_cv = pd.DataFrame(cv_records)

    # 7.2 Global Dimension Selection Curves (Repeated 3 x 5-Fold)
    sc_global = StandardScaler()
    x_matrix_sc = sc_global.fit_transform(x_matrix)
    global_pruned, global_vips = select_decorrelated_vip_features(x_matrix_sc, y_all, subset_feature_names)
    eval_max_m = min(MAX_PREDICTORS_EVAL, len(global_pruned))

    rkf_eval = RepeatedKFold(n_splits=5, n_repeats=3, random_state=RANDOM_STATE)
    curve_records = []

    for m in range(1, eval_max_m + 1):
        m_feats = global_pruned[:m]
        m_indices = [subset_feature_names.index(f_name) for f_name in m_feats]
        scores_m = []
        for tr, te in rkf_eval.split(x_matrix, y_all):
            sc_m = StandardScaler()
            x_tr_m = sc_m.fit_transform(x_matrix[tr][:, m_indices])
            x_te_m = sc_m.transform(x_matrix[te][:, m_indices])
            n_comps_m = 1 if m == 1 else (2 if m <= 3 else 3)
            pls_m = PLSRegression(n_components=n_comps_m).fit(x_tr_m, y_all[tr])
            scores_m.append(pls_m.score(x_te_m, y_all[te]))

        curve_records.append({
            'k_Variables': m,
            'R2_Mean': np.mean(scores_m),
            'R2_SE': np.std(scores_m) / np.sqrt(len(scores_m))
        })
    df_curve = pd.DataFrame(curve_records)

    # 1-SE Global Decision
    best_m_idx = df_curve['R2_Mean'].idxmax()
    thresh_global = df_curve.loc[best_m_idx, 'R2_Mean'] - df_curve.loc[best_m_idx, 'R2_SE']
    global_k_opt = int(df_curve[df_curve['R2_Mean'] >= thresh_global]['k_Variables'].min())

    # Final Fit across N total samples
    selected_k_features = global_pruned[:global_k_opt]
    opt_indices = [subset_feature_names.index(f_name) for f_name in selected_k_features]
    x_opt_sc = x_matrix_sc[:, opt_indices]

    inner_cv_glob = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    best_c_glob, max_score_glob = 1, -np.inf
    for c in range(1, min(global_k_opt, 5) + 1):
        c_scores = [PLSRegression(n_components=c).fit(x_opt_sc[t], y_all[t]).score(x_opt_sc[v], y_all[v])
                    for t, v in inner_cv_glob.split(x_opt_sc)]
        if np.mean(c_scores) > max_score_glob:
            max_score_glob = np.mean(c_scores)
            best_c_glob = c

    pls_final = PLSRegression(n_components=best_c_glob)
    pls_final.fit(x_opt_sc, y_all)
    y_pred_final = pls_final.predict(x_opt_sc).flatten()

    # Unstandardize coefficients for Google Earth Engine implementation
    beta_scaled = pls_final.coef_.flatten()
    means_opt = sc_global.mean_[opt_indices]
    scales_opt = sc_global.scale_[opt_indices]
    raw_coefficients = beta_scaled / scales_opt
    raw_intercept = np.mean(y_all) - np.dot(means_opt, raw_coefficients)

    # Physical metrics & feature summary table
    final_metrics = compute_regression_metrics(y_all, y_pred_final, global_k_opt)
    relative_importance = (np.abs(beta_scaled) / np.sum(np.abs(beta_scaled))) * 100.0
    vip_subset = [global_vips[subset_feature_names.index(f_name)] for f_name in selected_k_features]

    df_var_summary = pd.DataFrame({
        'Rank': range(1, global_k_opt + 1),
        'Selected_Feature': selected_k_features,
        'Standardized_Coeff': beta_scaled,
        'Raw_Coeff_GEE': raw_coefficients,
        'Contribution_Percent': relative_importance,
        'VIP_Score': vip_subset
    })

    df_perf_summary = pd.DataFrame({
        'Metric': ['R-squared (R2)', 'p-value', 'RMSE', 'MAE', 'PBIAS'],
        'Value': [final_metrics[3], final_metrics[4], final_metrics[0], final_metrics[1], final_metrics[2]],
        'Unit': ['-', '-', 'kg.DM/ha', 'kg.DM/ha', '%']
    })

    df_scatter = pd.DataFrame({'Observed': y_all, 'Predicted': y_pred_final})

    print(f"Optimal Variables Retained: {global_k_opt} | Performance: R2 = {final_metrics[3]:.3f}, RMSE = {final_metrics[0]:.2f}")

    return {
        'tag': model_tag,
        'cv_results': df_cv,
        'curve_results': df_curve,
        'var_table': df_var_summary,
        'perf_table': df_perf_summary,
        'scatter_data': df_scatter,
        'selected_features': selected_k_features,
        'raw_coefficients': raw_coefficients,
        'raw_intercept': raw_intercept
    }

# Execute pipelines across the three operational configurations
results_fused   = execute_pipeline(fused_features, "FUSED_S1_S2_ALOS_TOPO")
results_optical = execute_pipeline(optical_features, "OPTICAL_SENTINEL2")
results_radar   = execute_pipeline(radar_features, "RADAR_S1_ALOS")

# =========================================================================
# 8. MULTI-MODEL SYNTHESIS & REPORT GENERATION (EXCEL)
# =========================================================================
EXCEL_OUTPUT = 'Biomass_Modeling_MultiSensor_Synthesis.xlsx'
with pd.ExcelWriter(EXCEL_OUTPUT) as writer:
    for res in [results_fused, results_optical, results_radar]:
        prefix = res['tag'][:4]
        res['cv_results'].to_excel(writer, sheet_name=f"{prefix}_CV_Repeated", index=False)
        res['var_table'].to_excel(writer, sheet_name=f"{prefix}_Variables", index=False)
        res['curve_results'].to_excel(writer, sheet_name=f"{prefix}_R2_Curve", index=False)
        res['perf_table'].to_excel(writer, sheet_name=f"{prefix}_Metrics", index=False)
        res['scatter_data'].to_excel(writer, sheet_name=f"{prefix}_Predictions", index=False)

files.download(EXCEL_OUTPUT)

# =========================================================================
# 9. CARTOGRAPHY: HARMONIZED JENKS NATURAL BREAKS & CLOUD EXPORTS
# =========================================================================
print("\n>>> Building Cartographic Prediction Layers on Google Earth Engine...")

def construct_prediction_image(model_results):
    img = ee.Image.constant(float(model_results['raw_intercept']))
    for name, coef in zip(model_results['selected_features'], model_results['raw_coefficients']):
        img = img.add(master_hypercube.select(str(name)).multiply(float(coef)))
    return img.clamp(0, BIOMASS_CLAMP_MAX).rename('biomass_predicted').clip(roi)

# 9.1 Base continuous rasters
biomass_fused_img   = construct_prediction_image(results_fused)
biomass_optical_img = construct_prediction_image(results_optical)
biomass_radar_img   = construct_prediction_image(results_radar)

# 9.2 Compute Jenks Natural Breaks on the Master Fused Model
carto_sample = biomass_fused_img.sample(region=roi.geometry(), scale=30, numPixels=2500, seed=RANDOM_STATE)
kmeans_clusterer = ee.Clusterer.wekaKMeans(nClusters=N_CLASSES, distanceFunction='Euclidean', seed=RANDOM_STATE).train(carto_sample, ['biomass_predicted'])
fused_clustered = biomass_fused_img.cluster(kmeans_clusterer)

cluster_means = ee.List.sequence(0, N_CLASSES - 1).map(lambda c_id:
    ee.Feature(None, {
        'cluster_id': c_id,
        'mean': biomass_fused_img.updateMask(fused_clustered.eq(ee.Number(c_id))).reduceRegion(
            reducer=ee.Reducer.mean(), geometry=roi.geometry(), scale=50, maxPixels=1e8
        ).get('biomass_predicted')
    })
)
sorted_cluster_ids = ee.FeatureCollection(cluster_means).sort('mean').aggregate_array('cluster_id')
fused_jenks_class = fused_clustered.remap(sorted_cluster_ids, ee.List.sequence(0, N_CLASSES - 1)).clip(roi).toInt()

# Derive exact threshold break values from the Fused Jenks clustering
fused_thresholds = ee.List.sequence(0, N_CLASSES - 2).map(lambda c_id:
    biomass_fused_img.updateMask(fused_jenks_class.eq(ee.Number(c_id))).reduceRegion(
        reducer=ee.Reducer.max(), geometry=roi.geometry(), scale=50, maxPixels=1e8
    ).get('biomass_predicted')
).getInfo()

print(f"Extracted Harmonized Fused Jenks Break Thresholds: {fused_thresholds}")

# 9.3 Synchronize identical intervals across all models
def apply_fixed_thresholds(biomass_raster, breaks):
    classified = ee.Image(0)
    for class_idx, cut_value in enumerate(breaks):
        classified = classified.where(biomass_raster.gt(float(cut_value)), class_idx + 1)
    return classified.clip(roi).toInt()

class_fused   = fused_jenks_class
class_optical = apply_fixed_thresholds(biomass_optical_img, fused_thresholds)
class_radar   = apply_fixed_thresholds(biomass_radar_img, fused_thresholds)

# Formulate textual string labels for thematic vector attributes
label_bounds = [0] + [int(round(v)) for v in fused_thresholds]
text_labels = []
for i in range(len(label_bounds)):
    if i < len(label_bounds) - 1:
        text_labels.append(f"{label_bounds[i]} - {label_bounds[i+1]}")
    else:
        text_labels.append(f">= {label_bounds[i]}")

remap_keys = ee.List([str(i) for i in range(N_CLASSES)])
remap_vals = ee.List(text_labels)
label_dict = ee.Dictionary.fromLists(remap_keys, remap_vals)

def format_vector_polygons(feature):
    class_str = ee.Number(feature.get('label')).format('%d')
    return feature.set({
        'Classe_ID': ee.Number.parse(class_str),
        'Intervalle': label_dict.get(class_str)
    })

# 9.4 Task Export Submissions to Google Drive
models_dispatch = [
    ('Fused_S1_S2_Alos_Topo', biomass_fused_img, class_fused),
    ('Optical_Sentinel2', biomass_optical_img, class_optical),
    ('Radar_S1_Alos', biomass_radar_img, class_radar)
]

for label_str, continuous_img, class_img in models_dispatch:
    # Continuous GeoTIFF + Discrete Classification Band
    out_raster = continuous_img.addBands(class_img.rename('biomass_class')).toFloat()
    ee.batch.Export.image.toDrive(
        image=out_raster,
        description=f"Raster_Biomass_{label_str}",
        scale=10,
        region=roi.geometry(),
        fileFormat='GeoTIFF',
        maxPixels=1e9
    ).start()

    # Vector Polygons (Shapefile format with unified labels)
    out_vector = class_img.reduceToVectors(
        geometry=roi.geometry(),
        scale=20,
        geometryType='polygon',
        labelProperty='label',
        eightConnected=False
    ).map(format_vector_polygons).sort('Classe_ID', True)

    ee.batch.Export.table.toDrive(
        collection=out_vector,
        description=f"Vector_Biomass_{label_str}",
        fileFormat='SHP',
        selectors=['Classe_ID', 'Intervalle']
    ).start()

print("\n=========================================================================")
print("PIPELINE EXECUTED SUCCESSFULLY")
print(f"1. Model synthesis spreadsheet '{EXCEL_OUTPUT}' downloaded.")
print("2. Google Drive background export tasks generated.")
print("   Launch exports from the 'Tasks' tab in the GEE Code Editor or Colab.")
print("=========================================================================")